<a href="https://colab.research.google.com/github/luckiest-child/lab-4-llm-decision-support/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Afia Otieku-Boadu
**Student ID:** 55002028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [36]:
!pip install groq

In [37]:
import os
import json
import re
import pandas as pd

In [38]:
# --- Google Colab (Secrets panel) ---
from google.colab import userdata
GROQ_API_KEY = userdata.get("groqApi")

from groq import Groq

client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "llama-3.3-70b-versatile"

print("Client ready with Groq API and model:", MODEL_NAME)

print("Client ready.")

Client ready with Groq API and model: llama-3.3-70b-versatile
Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [39]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant that is concise and does not use filler words, or ask follow up questions unless required.", temperature=0.7, max_tokens=500):
      response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
          )
      return response

response = ask_llm("What is the best opening in chess?")
print(response.choices[0].message.content)
print("\nTokens consumed:", response.usage.total_tokens)
print(f"(Prompt tokens: {response.usage.prompt_tokens}, Completion tokens: {response.usage.completion_tokens})")

The best opening in chess is a matter of debate, but popular choices include:

1. e4 (King's Pawn Opening) - aggressive and open.
2. d4 (Queen's Pawn Opening) - solid and flexible.
3. Nf3 (Réti Opening) - strategic and positional.

These openings are often favored by grandmasters due to their versatility and ability to control the center of the board.

Tokens consumed: 150
(Prompt tokens: 66, Completion tokens: 84)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** 1. The system role tells the model how it should act, specifying its behaviour. eg. "You are a database consultant." The user role specifies the task the model is meant to perform. eg "Create an ER diagram that manages a digital school learning system".
2. A token is the smallest unit that a model can read and make sense out of. Each request varies in the number of tokens it uses and so billing by token handles the varied cost for each request as opposed to billing by requests which overlooks the resources consumed for each request.

### Part 1.2 — Temperature: the randomness dial

In [40]:
# Testing how temperature affects output
print("Temperature = 0.0: ") #using a temperature of 0.0
for i in range(5):
  print(i+1, ". ", ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature = 0.0).choices[0].message.content)

print("Temperature = 1.2: ") #using a temperature of 1.2
for i in range(5):
  print(i+1, ". ", ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature = 1.2).choices[0].message.content)

Temperature = 0.0: 
1 .  "Makola Save" - named after Makola Market, a major market in Accra.
2 .  "Makola Save" - named after Makola Market, a major market in Accra.
3 .  "Makola Save" - named after Makola Market, a major market in Accra.
4 .  "Makola Save" - named after Makola Market, a major market in Accra.
5 .  "Makola Save" - named after Makola Market, a major market in Accra.
Temperature = 1.2: 
1 .  MakolaSave.
2 .  "Makola Savings" or "Traders' Thrive" could work. "Makola" refers to a major market in Accra, creating a local connection.
3 .  MarketSafe or TraderSave could work, but a more local option might be 'Sika Box' (Sika means money in Ghanaian language, Akan).
4 .  "Sankofa Savings" or "Makola Fund" could work, as "Sankofa" is a Ghanaian symbol for looking back to plan for the future and "Makola" is a major market in Accra.
5 .  "Makola Save" - named after Makola Market, a major market in Accra.


**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** At temperature = 0.0, the model gave the exact same answer all 5 times but with temperature = 1.2, the answers varied each of the 5 times.  For a loan decision-support system, it will be more appropriate to use a temperature = 0 to ensure consistency and reliability rather than creativity with applications.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [41]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [42]:
# Testing how prompt affects the output
SUMMARY_PROMPT_V1 = "Summarize this:" #naive prompt attempt

SUMMARY_PROMPT_V2 = (
    "You are an assistant to a microfinance loan officer. Provide a concise, neutral, and factual summary of the loan application in exactly 3 to 4 sentences using only stated facts. Do not assume, or invent details."
) #prompt with a proper template

def summarize_v1(letter_text):
    prompt = f"{SUMMARY_PROMPT_V1}\n\n{letter_text}"
    return ask_llm(user_prompt=prompt).choices[0].message.content

def summarize_v2(letter_text):
    prompt = f"Summarize this loan application:\n\n{letter_text}"
    return ask_llm(user_prompt=prompt, system_prompt=SUMMARY_PROMPT_V2, temperature=0.0).choices[0].message.content

# Comparing V1 vs V2 outputs side by side.
print("L002:- ")
print("Prompt 1 (naive attempt) - ", summarize_v1(LETTERS["L002"]))
print("Prompt 2 (proper template) - ", summarize_v2(LETTERS["L002"]))
print()
print("L006:- ")
print("Prompt 1 (naive attempt) - ", summarize_v1(LETTERS["L006"]))
print("Prompt 2 (proper template) - ", summarize_v2(LETTERS["L006"]))

L002:- 
Prompt 1 (naive attempt) -  Kwame Boateng, a commercial driver, needs GHS 25,000 to repair his vehicle and pay debts. He promises to repay when business improves after the festive season, despite having no collateral.
Prompt 2 (proper template) -  Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng currently does not have collateral to offer. He expects to repay the loan when his business improves, anticipating an increase in income after the festive season.

L006:- 
Prompt 1 (naive attempt) -  Kofi, 22, is seeking GHS 50,000 to start 3 businesses: car washing, a provision shop, and importing phones from Dubai. He has no experience, but claims to be business-minded. He promises to repay the loan in 1 year with no collateral, relying on his trustworthiness.
Prompt 2 (proper template) -  Kofi has submitted a loan application for GHS 50,000 to fund three b

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** 1. V1's output did not include a lot of important details for making a decision as compared to V2's output which touched on details like the location, presence of collateral and others. 2. "No invented details" is an essential instruction because it ensures that the model does not make up missing or unknown information just to be able to satisfy the user. If it were not included, the model may invent false details about a user which may influence the final decision negatively. That is known as hallucination in LLM literature.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [43]:
EXTRACT_PROMPT = """You are a precise data extraction system for a microfinance institution.
Your task is to extract the specified details below from loan applications in the provided text into a valid JSON object.

Your output must match this schema exactly:
{
  "applicant_name": string or null,
  "amount_ghs": number or null,
  "purpose": string or null,
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean or null,
  "repayment_months": number or null
}

RULES:
1. Output MUST be strictly raw JSON without markdown backticks, conversational preamble, or explanations.
2. If a field is not explicitly stated in the letter, set its value to null. Do not guess or estimate.
3. For "has_collateral_or_guarantor", return true if collateral, savings pledge, or a guarantor is explicitly mentioned; return false if explicitly stated they have none; return null only if entirely unmentioned.
4. "amount_ghs", "monthly_profit_ghs", and "repayment_months" must be purely numeric (e.g. 8000, not "GHS 8,000").

Example:
Input Letter:
"Hello, I am Kwabena Mensah from Kasoa. I run a shoe repair kiosk. I need a loan of GHS 4,000 to buy leather and soling sheets. I have no guarantor or assets to pledge. I make GHS 600 monthly and can pay back over 8 months."

Output:
{
  "applicant_name": "Kwabena Mensah",
  "amount_ghs": 4000,
  "purpose": "buy leather and soling sheets",
  "monthly_profit_ghs": 600,
  "has_collateral_or_guarantor": false,
  "repayment_months": 8
}
"""

def extract_fields(letter_text, temperature=0.0):
  # Calls the LLM, strips any ```json fences, json.loads() the result, and returns a dict. Handle parse failures.
    """
    Calls LLM to extract JSON data and handles fences and formatting errors cleanly.
    """
    user_prompt = f"Extract structured loan information from this application:\n\n{letter_text}"
    response = ask_llm(
        user_prompt=user_prompt,
        system_prompt=EXTRACT_PROMPT,
        temperature=temperature,
        max_tokens=400
    )
    raw_content = response.choices[0].message.content.strip()

    # Strip markdown backticks if present
    cleaned = re.sub(r"^```(?:json)?\s*", "", raw_content, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        data = json.loads(cleaned)
        return data
    except json.JSONDecodeError as e:
        print(f"Warning: JSON parse failed ({e}). Raw response was:\n{raw_content}")
        return None

# Storing the results in a DataFrame and displaying it
records = []
for id, letter in LETTERS.items():
    extracted = extract_fields(letter)
    if extracted:
        extracted["letter_id"] = id
        records.append(extracted)

df_extracted = pd.DataFrame(records).set_index("letter_id")[
    ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]
]
display(df_extracted)

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** 1. If the few-shot example came from the letters it may cause data leakage. Instead of the model learning the format it would rather learn the information to be outputted which may make it difficult to generalise on unseen data.
2. Using "not null, do not guess" prevents the model from hallucinating and creating values that do not exist in order to fill in missing values.
3. Temperature = 0 prevents the model from being creative with the information and ensures that it sticks to the facts and format. It ensures that the model is consistent and reliable always in its output. If the task were to be a creative one then the temperature will have to be greater than 0 to increase its randomness.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [44]:
# Prompt to summarise the important details of an application and suggest next steps
BRIEF_PROMPT = """You are an expert credit risk analyst assistant to a loan officer at a Ghanaian microfinance institution.
Your task is to provide a concise objective decision-support brief.

CRITICAL POLICY:
1. You are NOT to make the final approval or rejection decision. Final credit approval is strictly reserved for the loan officer.
2. Frame all assessments as decision support.
3. Your output must follow this exact structured format:

- Strengths as bullet points of concrete, grounded facts indicating repayment capacity, stability, collateral, or track record
- Risks / Red Flags as bullet points of financial risks, unverified claims, missing security, or cash flow concerns
- Missing Information (Specific documentation or facts the loan officer must obtain before making a decision)
- Suggested Next Step (e.g., "Invite for in-person interview", "Request bank/sales statements", "Conduct site visit to verify stall", "Flag for senior credit committee review")
"""

def generate_brief(letter_text, extracted_json):
    user_prompt = f"""Generate a decision-support brief for the following loan application:

\"\"\"{letter_text}\"\"\"

Extracted Data:
{json.dumps(extracted_json, indent=2)}
"""
    response = ask_llm(user_prompt=user_prompt, system_prompt=BRIEF_PROMPT, temperature=0.0, max_tokens=600)
    return response.choices[0].message.content

# Generate briefs for all letters and display for L001, L002, L003 and L006
briefs = {}
for id in LETTERS:
    briefs[id] = generate_brief(LETTERS[id], df_extracted.loc[id].to_dict())

for id in ["L001", "L002", "L003", "L006"]:
    print(f"Decision Support Brief: \n{id}: ")
    print(briefs[id])
    print("\n")

Decision Support Brief: 
L001: 
- Strengths:
  * 12 years of experience selling provisions at Makola Market, indicating a stable business history
  * Consistent profit of GHS 900 per month from the current stall, demonstrating a reliable income stream
  * GHS 2,500 savings with the susu scheme over two years, showing a track record of savings discipline and commitment
  * A guarantor, the applicant's sister, who is a teacher, potentially providing an additional layer of repayment security
  * Proposed monthly repayment of GHS 450, which is approximately half of the monthly profit, suggesting a manageable repayment plan

- Risks / Red Flags:
  * Expansion into frozen foods may introduce new business risks, such as increased electricity costs, spoilage, and competition
  * The deep freezer purchase may require additional, unforeseen expenses, such as maintenance and repairs
  * No detailed financial statements or records of the business are provided for review
  * The guarantor's financi

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** 1. The model did well to identify the right strengths and red flags in both L003 and L006.
2. There may be external factors that may lead to the approval or rejection of one application as compared to another that the model may not have accss to and so it will not be a good idea to allow the model to approve or reject an application. For example, if there is extra information about the personal details of a person that may influence the final decision, it will not be ethical to feed all that information into the model as a measure to protecting the privacy of individuals.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** c5528ee

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [45]:
fields_to_eval = [
    "applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
    "has_collateral_or_guarantor", "repayment_months"
]

def evaluate_field(pred, gold):
    # Handle cases where gold is None (meaning missing in gold standard)
    if gold is None:
        return pd.isna(pred)

    # Handle boolean comparisons
    if isinstance(gold, bool):
        return pred == gold

    # Handle numeric comparisons (int or float)
    if isinstance(gold, (int, float)):
        if pd.isna(pred):
            return False
        try:
            return float(pred) == float(gold)
        except (TypeError, ValueError):
            return False

    # Handle string comparisons (keeping original user's logic that passed previous tests)
    if isinstance(gold, str):
        if pred is None:
            return False
        return any(term.lower() in str(pred).lower() for term in gold.split() if len(term) > 3)

    return pred == gold

accuracy_rows = []
for field in fields_to_eval:
    row = {"Field": field}
    matches = []
    for id in ["L001", "L003", "L006"]:
        pred_val = df_extracted.loc[id, field]
        gold_val = GOLD[id][field]
        is_match = evaluate_field(pred_val, gold_val)
        row[id] = f"Pred: {pred_val} | Gold: {gold_val} ({'PASS' if is_match else 'FAIL'})"
        matches.append(1 if is_match else 0)
    row["Accuracy"] = f"{(sum(matches)/len(matches))*100:.1f}%"
    accuracy_rows.append(row)

acc_table = pd.DataFrame(accuracy_rows).set_index("Field")
display(acc_table)

,L001,L003,L006,Accuracy
Field,,,,
applicant_name,Pred: Akosua Mensah | Gold: Akosua Mensah (PASS),Pred: Efua Darko | Gold: Efua Darko (PASS),Pred: Kofi | Gold: Kofi (PASS),100.0%
amount_ghs,Pred: 8000 | Gold: 8000 (PASS),Pred: 15000 | Gold: 15000 (PASS),Pred: 50000 | Gold: 50000 (PASS),100.0%
purpose,Pred: buy a deep freezer and expand into froze...,Pred: purchase two industrial sewing machines ...,"Pred: start a car washing business, a provisio...",100.0%
monthly_profit_ghs,Pred: 900.0 | Gold: 900 (PASS),Pred: 2800.0 | Gold: 2800 (PASS),Pred: nan | Gold: None (PASS),100.0%
has_collateral_or_guarantor,Pred: True | Gold: True (PASS),Pred: True | Gold: True (PASS),Pred: False | Gold: False (PASS),100.0%
repayment_months,Pred: 20.0 | Gold: 20 (PASS),Pred: 15.0 | Gold: 15 (PASS),Pred: 12.0 | Gold: 12 (PASS),100.0%


### Part 4.2 — Reliability: is the system consistent?

In [46]:
# Running extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at temperature=1.0 and reporting the number of runs producing (a) valid JSON and (b) identical values across runs.

def run_reliability_experiment(letter_text, runs=5):
    results = {0.0: [], 1.0: []}
    for temp in [0.0, 1.0]:
        for i in range(runs):
            data = extract_fields(letter_text, temperature=temp)
            results[temp].append(data)
    return results

reliability_results = run_reliability_experiment(LETTERS["L004"], runs=5)

for temp, res_list in reliability_results.items():
    valid_json_count = sum(1 for r in res_list if r is not None)
    serialized = [json.dumps(r, sort_keys=True) for r in res_list if r is not None]
    unique_outputs_count = len(set(serialized))
    print(f"Temperature {temp}:")
    print(f"  - Valid JSON outputs: {valid_json_count} / {len(res_list)}")
    print(f"  - Unique distinct outputs: {unique_outputs_count}")

Temperature 0.0:
  - Valid JSON outputs: 5 / 5
  - Unique distinct outputs: 1
Temperature 1.0:
  - Valid JSON outputs: 5 / 5
  - Unique distinct outputs: 1


### Part 4.3 — Hallucination probing

In [47]:
# Conducting adversarial tests
# Asking for a detail that is not present in the application letter
adversarial_query_1 = (
    "What is the applicant's credit score based on the following application letter?\n\n"
    f"{LETTERS['L001']}"
)

response_1 = ask_llm(
    adversarial_query_1,
    system_prompt="You are a factual assistant to a credit officer. If information is not in the text, clearly state that it is not provided. Do not guess or make up values.",
    temperature=0.0
).choices[0].message.content

test_1 = ("not" in response_1.lower() or "unmentioned" in response_1.lower() or "does not" in response_1.lower()) and not re.search(r'\b[3-8][0-9]{2}\b', response_1)

# Feeding the model an irrelevant text and testing for hallucination
irrelevant_text = "Final exams begin on the 24th of August and end on the 28th of the same month."
response_2 = extract_fields(irrelevant_text, temperature=0.0)

test_2 = response_2 is not None and all(v is None for v in response_2.values())

print(f"Test 1 (Unmentioned Credit Score):\nResponse: {response_1}\nVerdict: {'PASS' if test_1 else 'FAIL'}\n")
print(f"Test 2 (Irrelevant Text):\nResponse: {json.dumps(response_2, indent=2)}\nVerdict: {'PASS' if test_2 else 'FAIL'}")

Test 1 (Unmentioned Credit Score):
Response: The applicant's credit score is not provided in the application letter. The letter includes information about the applicant's business, income, savings, and proposed loan repayment plan, but it does not mention a credit score.
Verdict: PASS

Test 2 (Irrelevant Text):
Response: {
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": null,
  "repayment_months": null
}
Verdict: PASS


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** 1. The model recorded 100% extraction accuracy.
2. Since the model was working with fixed data and facts, the temperature did not affect the output of the model as it was told not to fabricate values.
3. The system did not hallucinate under probing.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** 1. If the bank fully automated decisions, applicants with poor English or illiterate applicants who may have difficulty producing the right requirements may be rejected and harmed in the process.
2. Sending letters with personal data is harmful to the applicants as their data could be stolen and used against them. Before deploying such a system at a real institution, we must check that all the personal details of applicants are kept protected and handled by trusted workers only. Only information needed by the system should be fed into it.
3. In deploying this system, there is a need for human review and monitoring of the system. The humans are to review everything the system produces to ensure consistency and reliability while the monitoring will be to ensure that the system is not hallucinating or giving false values, and to correct it when it goes wrong.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** 1. Both iterating on a prompt and iterating on model parameters affect the efficiency of the output. A bad promt/parameter produces a sub-efficient output while a good prompt/parameter produces an efficient output. Iterating on a prompt is mostly textual while iterating on a parameter varies in input definition.
2. I will trust the system but only if there is regular monitoring and oversight by humans. The outputs from the tests were consistent with the information in the data and so we can conclude that the system is reliable.
3. In processing the initial naive prompt, approximately 150 tokens were needed. In processing the prompts with the proper template it will cost much more and doing so for a 1,000 applications will cost a lot to tokenize all those applications. In choosing a provider, one should focus on cost as well as reliability and quality.
4. APIs are trained on more data and so are more efficient than training my own model which will only have access to limited data. It will be better to train my own model than use an API when I am trying to perform tasks based on a very specific context such as a model that requires information about my personal daily activities.